In [6]:
"""
ЗАДАНИЕ 3
3.1. Реализуйте интерактивный режим:
     - add <слово>
     - check <слово>  -> yes/no

3.2. Возьмите длинный текст, добавьте каждое слово в словарь и сравните
     число коллизий с "идеальной" равномерной хеш-функцией.
"""

import unicodedata
from dataclasses import dataclass
from typing import List, Iterable


# ---------------------------
# ХЕШ-ФУНКЦИЯ (djb2 модернизированная)
# ---------------------------

MASK64 = 0xFFFFFFFFFFFFFFFF  # ограничение вычислений 64 битами (uint64)

def normalize_word(word: str) -> str:
    """
    Нормализация слова для естественного языка:
    - NFC: приводит Unicode к каноническому виду
    - casefold(): регистронезависимость (лучше, чем lower для Unicode)
    """
    return unicodedata.normalize("NFC", word).casefold()


def fmix64(k: int) -> int:
    """
    Финальный bit-mixer (finalizer) для "лавинного эффекта":
    хорошо перемешивает биты промежуточного хеша.
    """
    k &= MASK64
    k ^= (k >> 33) & MASK64
    k = (k * 0xff51afd7ed558ccd) & MASK64
    k ^= (k >> 33) & MASK64
    k = (k * 0xc4ceb9fe1a85ec53) & MASK64
    k ^= (k >> 33) & MASK64
    return k & MASK64


def djb2_plus(word: str) -> int:
    """
    Модернизированный djb2 (djb2a + finalizer):
    1) normalize: NFC + casefold
    2) UTF-8 bytes
    3) djb2a: h = h*33 ^ byte
    4) 64-bit arithmetic
    5) примешиваем длину
    6) финальный mixer fmix64
    """
    w = normalize_word(word)
    data = w.encode("utf-8")

    h = 5381  # классический seed djb2
    for b in data:
        h = ((h * 33) ^ b) & MASK64  # djb2a (XOR-версия)

    h ^= len(data)
    return fmix64(h)


# ---------------------------
# ХЕШ-ТАБЛИЦА (separate chaining)
# ---------------------------

@dataclass
class HashSetStats:
    # "Коллизии при вставке": сколько элементов уже было в бакете, когда вставляли новое слово
    collisions: int = 0
    inserts: int = 0
    duplicates: int = 0


class HashSetChaining:
    """
    HashSet слов через separate chaining:
    - buckets: список списков
    - при коллизии элементы оказываются в одном бакете (в списке)
    """

    def __init__(self, initial_capacity: int = 1024, max_load: float = 0.75):
        # делаем capacity степенью двойки (удобно индекс = hash & (capacity-1))
        cap = 1
        while cap < initial_capacity:
            cap <<= 1

        self.capacity = cap
        self.mask = self.capacity - 1
        self.buckets: List[List[str]] = [[] for _ in range(self.capacity)]
        self.size = 0
        self.max_load = max_load
        self.stats = HashSetStats()

    def _bucket_index(self, normalized_word: str) -> int:
        # индекс бакета по низшим битам хеша
        return djb2_plus(normalized_word) & self.mask

    def _rehash(self) -> None:
        """
        Увеличиваем таблицу в 2 раза и перераскладываем слова.
        Это нужно, чтобы средняя длина бакета не росла слишком сильно.
        """
        old_buckets = self.buckets

        self.capacity <<= 1
        self.mask = self.capacity - 1
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0

        for bucket in old_buckets:
            for w in bucket:
                idx = self._bucket_index(w)
                self.buckets[idx].append(w)
                self.size += 1

    def add(self, word: str) -> None:
        """
        add <слово>
        - нормализуем слово
        - если оно уже есть: duplicates++
        - иначе: добавляем и увеличиваем счётчик коллизий на len(bucket)
        """
        w = normalize_word(word)

        # контроль load factor
        if self.size + 1 > int(self.capacity * self.max_load):
            self._rehash()

        idx = self._bucket_index(w)
        bucket = self.buckets[idx]

        # проверка на наличие (чтобы не хранить дубликаты)
        for x in bucket:
            if x == w:
                self.stats.duplicates += 1
                return

        # новое слово -> считаем "вставочные" коллизии как размер бакета до вставки
        self.stats.collisions += len(bucket)

        bucket.append(w)
        self.size += 1
        self.stats.inserts += 1

    def contains(self, word: str) -> bool:
        """
        check <слово> -> True/False
        """
        w = normalize_word(word)
        idx = self._bucket_index(w)
        return any(x == w for x in self.buckets[idx])

    def bucket_sizes(self) -> List[int]:
        """
        Размеры бакетов (для анализа распределения)
        """
        return [len(b) for b in self.buckets]


# ---------------------------
# 3.1. ИНТЕРАКТИВНЫЙ РЕЖИМ (REPL)
# ---------------------------

def run_repl() -> HashSetChaining:
    """
    Команды:
      add <слово>
      check <слово>   -> yes/no
      exit            -> выход
    """
    hs = HashSetChaining(initial_capacity=1024)
    print("Команды: add <слово> | check <слово> | exit")

    while True:
        try:
            line = input().strip()
        except EOFError:
            break

        if not line or line.lower() == "exit":
            break

        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            print("Неверная команда")
            continue

        cmd, word = parts[0], parts[1]

        if cmd == "add":
            hs.add(word)
        elif cmd == "check":
            print("yes" if hs.contains(word) else "no")
        else:
            print("Неверная команда")

    return hs


# ---------------------------
# 3.2. ЭКСПЕРИМЕНТ КОЛЛИЗИЙ + "ИДЕАЛ"
# ---------------------------

def text_to_words_letters_only(text: str) -> List[str]:
    """
    Разбор текста в слова:
    - слово = последовательность букв (isalpha)
    - всё небуквенное игнорируем/считаем разделителем
    - сохраняем порядок
    - приводим к casefold
    """
    words: List[str] = []
    buf: List[str] = []

    for ch in text:
        if ch.isalpha():
            buf.append(ch.casefold())
        else:
            if buf:
                words.append("".join(buf))
                buf.clear()

    if buf:
        words.append("".join(buf))

    return words


def observed_collisions_pairs(bucket_sizes: Iterable[int]) -> int:
    """
    Коллизии как число "пар" в одном бакете:
    sum C(k,2) = sum k*(k-1)/2
    """
    total = 0
    for k in bucket_sizes:
        total += k * (k - 1) // 2
    return total


def ideal_collisions_pairs(n_unique: int, m_buckets: int) -> int:
    """
    Идеально равномерное распределение n элементов по m бакетам:
    q = n // m
    r = n % m
    r бакетов размера q+1, остальные m-r бакетов размера q
    """
    q, r = divmod(n_unique, m_buckets)
    return r * ((q + 1) * q // 2) + (m_buckets - r) * (q * (q - 1) // 2)


def collision_experiment_from_text(text: str, initial_capacity: int = 4096) -> dict:
    """
    Добавляем все слова из текста в HashSetChaining и считаем:
    - insert_collisions: "вставочные" коллизии (len(bucket) при добавлении нового слова)
    - observed_pairs: sum C(k,2) по бакетам (фактическая "плотность" коллизий)
    - ideal_pairs: идеал при равномерном распределении
    """
    hs = HashSetChaining(initial_capacity=initial_capacity)

    for w in text_to_words_letters_only(text):
        hs.add(w)

    sizes = hs.bucket_sizes()
    obs_pairs = observed_collisions_pairs(sizes)
    ideal_pairs = ideal_collisions_pairs(hs.size, hs.capacity)

    return {
        "unique_words": hs.size,
        "capacity": hs.capacity,
        "insert_collisions": hs.stats.collisions,
        "duplicates_add": hs.stats.duplicates,
        "observed_pairs": obs_pairs,
        "ideal_pairs": ideal_pairs,
        "ratio_observed_to_ideal": (obs_pairs / ideal_pairs) if ideal_pairs else 0.0,
    }


# ---------------------------
# ГОТОВЫЕ ПРИМЕРЫ ЗАПУСКА (как в задаче 1)
# ---------------------------

# ===== Пример 1: симуляция команд add/check =====
commands = [
    "add Привет",
    "add мир",
    "add Мир",          # после normalize_word это то же слово, что и "мир"
    "check привет",
    "check МИР",
    "check python",
    "add python",
    "check python",
]

hs = HashSetChaining(initial_capacity=16)

print("Команды:")
for cmd in commands:
    print(" ", cmd)

print("\nОтветы на check:")
for line in commands:
    cmd, word = line.split(maxsplit=1)
    if cmd == "add":
        hs.add(word)
    elif cmd == "check":
        print("yes" if hs.contains(word) else "no")

print("\nСтатистика:")
print("  Уникальных слов в словаре:", hs.size)
print("  Коллизии (при вставках):", hs.stats.collisions)
print("  Дубликаты add:", hs.stats.duplicates)
print("  Ёмкость (бакетов):", hs.capacity)

# ===== Пример 2: запуск интерактивного режима =====
# hs = run_repl()

# ===== Пример 3: эксперимент 3.2 на файле input.txt =====
with open("input.txt", "r", encoding="utf-8") as f:
     text = f.read()
print(collision_experiment_from_text(text, initial_capacity=4096))

Команды:
  add Привет
  add мир
  add Мир
  check привет
  check МИР
  check python
  add python
  check python

Ответы на check:
yes
yes
no
yes

Статистика:
  Уникальных слов в словаре: 3
  Коллизии (при вставках): 0
  Дубликаты add: 1
  Ёмкость (бакетов): 16
{'unique_words': 209, 'capacity': 4096, 'insert_collisions': 4, 'duplicates_add': 71, 'observed_pairs': 4, 'ideal_pairs': 0, 'ratio_observed_to_ideal': 0.0}
